In [1]:
import numpy as np 
import matplotlib as plt
import pandas as pd 
import yfinance as yf 
import talib as ta
import datetime as dt 
import statsmodels.api as sm

import warnings
warnings.filterwarnings('ignore')

In [2]:
# data 
ticker1 = yf.Ticker('TD')
ticker2 = yf.Ticker('RY')

training_start_date = dt.datetime(2018,1,1)
training_end_date = dt.datetime(2022,1,1)
testing_start_date = training_end_date
testing_end_date = dt.datetime.now()

P1 = ticker1.history(start=training_start_date, end=testing_end_date)
P2 = ticker2.history(start=training_start_date, end=testing_end_date)

# Get ln prices so that we look at % change not absolute $ change
P1['lnClose'] = np.log(P1['Close'])
P2['lnClose'] = np.log(P2['Close'])


In [5]:
# build df with relevant data -> date, lnP1, lnP2

df = pd.DataFrame({
    "P1": P1["lnClose"].values, # get values from ln columns
    "P2": P2["lnClose"].values,
})
df = df.set_index(P1.index)

# Remove timezone data from date array
df.index = pd.to_datetime(df.index).tz_localize(None)

df

,P1,P2
Date,,
2018-01-02,3.733357,4.101831
2018-01-03,3.737761,4.108007
2018-01-04,3.748853,4.120960
2018-01-05,3.757507,4.126663
2018-01-08,3.755019,4.122270
...,...,...
2026-06-30,4.799338,5.332574
2026-07-01,4.806068,5.339027
2026-07-02,4.781641,5.321985


In [16]:
# Regression on training data
# lnP1 = alpha + beta * lnP2 + residual

# Get price data within training period
train = df.loc[training_start_date:training_end_date].copy()

x_train = train["P1"]
y_train = train["P2"]

# Add intercept term column, else forced intercept at origin -> [const, P1]
x_train = sm.add_constant(x_train)

# fit Ordinary Least Squares model on x and y data -> returns value for const and slope 
model = sm.OLS(y_train, x_train).fit()

# assign intercept alpha and slope beta 
alpha = model.params["const"] # intercept
beta = model.params["P1"] # slope

print("alpha =", alpha)
print("beta =", beta)
print(model.summary())

alpha = 0.5249015306054758
beta = 0.9623339869693233
                            OLS Regression Results                            
Dep. Variable:                     P2   R-squared:                       0.908
Model:                            OLS   Adj. R-squared:                  0.908
Method:                 Least Squares   F-statistic:                     9917.
Date:                Tue, 07 Jul 2026   Prob (F-statistic):               0.00
Time:                        15:31:01   Log-Likelihood:                 1567.5
No. Observations:                1008   AIC:                            -3131.
Df Residuals:                    1006   BIC:                            -3121.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------